# Data preprocessing

| Dataset | Contents | Fields and units |
|:---|:---|:---|
| **ECA&D stations** | Reported temperature maxima, with one record per station series and date. | `TX` in 0.1°C, `DATE`, station ID `STAID`, source ID `SOUID`, quality flag `Q_TX`. |
| **ERA5-Land** | Hourly 2 m air temperature on a 0.1° grid. | `t2m` in Kelvin, UTC `valid_time`, `latitude`, `longitude`. |

The existing daily ERA5 file below supplies the grid coordinates.

In [1]:
import polars as pl
import xarray as xr
from IPython.display import display

from py.paths import AnalysisDataDirectory as data_dir
from py.paths import RawDataDirectory as raw_data_dir

station_data = pl.read_parquet(data_dir / "station_data.parquet")
station_data.head(5)

STAID,SOUID,DATE,TX,Q_TX
i32,i32,i32,i16,i8
1,35382,18820101,25,0
1,35382,18820102,37,0
1,35382,18820103,60,0
1,35382,18820104,55,0
1,35382,18820105,10,0


## 1. Station quality

Keep provider-valid records (`Q_TX == 0`) with a reported temperature (`TX != -9999`). Apply this per daily observation.

In [2]:
valid_station_data = station_data.filter(
    (pl.col("Q_TX") == 0) & (pl.col("TX") != -9999)
)

quality_counts = pl.DataFrame([
    {"sample": name, "station_series": frame["STAID"].n_unique(), "daily_records": frame.height}
    for name, frame in [("Raw", station_data), ("Provider-valid", valid_station_data)]
])
quality_counts

sample,station_series,daily_records
str,i64,i64
"""Raw""",8568,175615474
"""Provider-valid""",8568,133209364


## 2. Geographic selection

Keep **European observation sources inside the ERA5 extract**:

- Country list: [UN M49 Europe](https://unstats.un.org/unsd/methodology/m49/overview/), including Iceland and the Nordic countries; excluding Türkiye, Cyprus and Greenland.
- Coverage: **34.95–69.95°N, 25.05°W–40.95°E**, measured at the outer cell edges.

Use the source's country and coordinates because blended station series can borrow observations from other locations.

In [3]:
EUROPEAN_COUNTRY_CODES = """
AL AD AT AX BY BE BA BG HR CZ DK EE FO FI FR DE GI GR GG VA HU IS IE IM
IT JE LV LI LT LU MT MD MC ME NL MK NO PL PT RO RU UA SM RS SK SI ES SJ SE
CH GB
""".split()

era5_file = raw_data_dir / "derived-era5-land-daily-statistics/2020_2m_temperature_daily_maximum.nc"
with xr.open_dataset(era5_file) as grid:
    latitudes = grid.latitude.values
    longitudes = grid.longitude.values
    half_lat = abs(float(latitudes[1] - latitudes[0])) / 2
    half_lon = abs(float(longitudes[1] - longitudes[0])) / 2
    south, north = round(float(latitudes.min()) - half_lat, 6), round(float(latitudes.max()) + half_lat, 6)
    west, east = round(float(longitudes.min()) - half_lon, 6), round(float(longitudes.max()) + half_lon, 6)

In [4]:
from py.geographic_map import geographic_selection_map
from py.paths import NotebookFiguresDirectory as notebook_figures_dir
from py.publication import SaveFigure as save_figure

geography_map = geographic_selection_map(EUROPEAN_COUNTRY_CODES, (west, south, east, north))
map_dir = notebook_figures_dir / "preprocessing_walkthrough"
save_figure(geography_map, map_dir / "geographic_selection", WidthMm=240, Dpi=150)
display(geography_map)

alt.LayerChart(...)

Blue: eligible European land. Pale blue: European land outside the extract. Gray: other countries. Orange dashed line: ERA5 boundary.

[station_metadata.py](../station_metadata.py) reads `sources.txt` and resolves the supplied source-ID mapping: participant metadata IDs have a +100000 offset; synoptic IDs match directly.

In [5]:
from py.preprocessing.station_metadata import attach_source_locations

observation_sources = valid_station_data.group_by("STAID", "SOUID").len(name="observations")
source_geography = attach_source_locations(observation_sources)

european_sources = source_geography.filter(pl.col("country_code").is_in(EUROPEAN_COUNTRY_CODES))
selected_sources = european_sources.filter(
    pl.col("source_lat").is_between(south, north)
    & pl.col("source_lon").is_between(west, east)
)
selected_station_data = valid_station_data.join(
    selected_sources.select("STAID", "SOUID"), on=["STAID", "SOUID"], how="semi"
)

geography_counts = pl.DataFrame([
    {
        "sample": name,
        "station_series": frame["STAID"].n_unique(),
        "daily_records": frame["observations"].sum(),
    }
    for name, frame in [("Europe", european_sources), ("Europe within ERA5", selected_sources)]
])
geography_counts

sample,station_series,daily_records
str,i64,i64
"""Europe""",8204,127711784
"""Europe within ERA5""",7515,113368058


## 3. Isolated spikes and dips

Exclude a daily maximum **at least 25°C above both adjacent days, or at least 25°C below both** ([Durre et al., 2010](https://www.ncei.noaa.gov/pub/data/ghcn/daily/papers/durre-menne-etal2010.pdf)).

Check consecutive calendar days within the same `SOUID`, counting borrowed observations once. Both neighbours must be present. Compute flags once, before the range filter; remove every copy of a flagged source-day, keeping the other days.

In [6]:
source_days = (
    selected_station_data.select("SOUID", "DATE", "TX")
    .unique()
    .with_columns(pl.col("DATE").cast(pl.String).str.to_date("%Y%m%d").alias("date"))
    .sort("SOUID", "date")
)

daily_context = source_days.with_columns(
    pl.col("TX").shift(1).over("SOUID").alias("previous_tx"),
    pl.col("TX").shift(-1).over("SOUID").alias("next_tx"),
    (
        (pl.col("date").diff().over("SOUID").dt.total_days() == 1)
        & (pl.col("date").diff(-1).over("SOUID").dt.total_days() == -1)
    ).fill_null(False).alias("testable"),
)

above_previous = pl.col("TX") - pl.col("previous_tx")
above_next = pl.col("TX") - pl.col("next_tx")
spike_days = daily_context.filter(
    pl.col("testable")
    & (
        ((above_previous >= 250) & (above_next >= 250))
        | ((above_previous <= -250) & (above_next <= -250))
    )
)

spike_keys = spike_days.select("SOUID", "DATE")
spike_observations = selected_station_data.join(spike_keys, on=["SOUID", "DATE"], how="semi")
spike_filtered_station_data = selected_station_data.join(
    spike_keys, on=["SOUID", "DATE"], how="anti"
)

pl.DataFrame({
    "selected_records": [selected_station_data.height],
    "testable_source_days": [daily_context["testable"].sum()],
    "flagged_source_days": [spike_days.height],
    "excluded_records": [spike_observations.height],
    "retained_records": [spike_filtered_station_data.height],
})

selected_records,testable_source_days,flagged_source_days,excluded_records,retained_records
i64,i64,i64,i64,i64
113368058,80271165,46,69,113367989


In [7]:
spike_days.select(
    "SOUID", "date",
    (pl.col("previous_tx") / 10).alias("previous_celsius"),
    (pl.col("TX") / 10).alias("spike_celsius"),
    (pl.col("next_tx") / 10).alias("next_celsius"),
).head(10)

SOUID,date,previous_celsius,spike_celsius,next_celsius
i32,date,f64,f64,f64
934,1982-04-15,6.6,36.3,4.6
5242,1995-01-04,7.4,50.0,1.0
12218,2003-05-07,27.8,0.0,29.0
15586,2004-07-08,29.8,0.0,31.2
37305,1961-12-11,-10.2,-35.2,-8.6
37422,1978-12-24,-9.0,-35.0,-4.0
37476,1902-12-02,-9.0,-34.0,-3.0
55824,1998-09-01,28.2,0.0,29.6
55824,1998-09-04,28.4,0.0,27.8


## 4. Temperature bounds

After removing spike days, keep daily maxima from **−60°C to 50°C**, inclusive. Exclude observations outside these provisional bounds; retain their original values in the exclusion table.

In [8]:
MIN_TEMPERATURE_C = -60
MAX_TEMPERATURE_C = 50

in_temperature_range = pl.col("TX").is_between(
    10 * MIN_TEMPERATURE_C, 10 * MAX_TEMPERATURE_C
)
range_exclusions = spike_filtered_station_data.filter(~in_temperature_range)
clean_station_data = spike_filtered_station_data.filter(in_temperature_range)

pl.DataFrame({
    "after_spike_filter": [spike_filtered_station_data.height],
    "excluded_below_minimum": [range_exclusions.filter(pl.col("TX") < 10 * MIN_TEMPERATURE_C).height],
    "excluded_above_maximum": [range_exclusions.filter(pl.col("TX") > 10 * MAX_TEMPERATURE_C).height],
    "retained_records": [clean_station_data.height],
})

after_spike_filter,excluded_below_minimum,excluded_above_maximum,retained_records
i64,i64,i64,i64
113367989,1,322,113367666


## 5. Study period

Keep observation dates from **1 January 2020 to 31 August 2025**, matching the hourly ERA5 download. Spike flags above use adjacent days before this date restriction.

In [9]:
import json

audit_dir = data_dir / "cleaning_audit"
era5_plan = json.loads((audit_dir / "era5_hourly_plan.json").read_text())
era5_first_date = int(era5_plan["start_utc"][:10].replace("-", ""))
era5_last_date = int(era5_plan["end_utc_inclusive"][:10].replace("-", ""))

study_station_data = clean_station_data.filter(
    pl.col("DATE").is_between(era5_first_date, era5_last_date)
)
pl.DataFrame({
    "first_date": [study_station_data["DATE"].min()],
    "last_date": [study_station_data["DATE"].max()],
    "daily_records": [study_station_data.height],
    "station_series": [study_station_data["STAID"].n_unique()],
})

first_date,last_date,daily_records,station_series
i64,i64,i64,i64
20200101,20250831,9840278,5613


## 6. Observation windows in UTC

Map **2020–August 2025 observations** using each source's country, provider and `ELEI` code. The [verified rules](../../../data2/cleaning_audit/verified_observation_window_rules.csv) record their supporting URLs and applicable dates.

- Verified `TX2`, modern Finnish `TX1` and Austrian `TX20`: previous day 18:00 → labelled day 18:00 UTC.
- Slovenian `TX8` and verified Serbian `TX8` in 2021: previous day 20:00 → labelled day 20:00 UTC.
- French `TX6`: labelled day 06:00 → next day 06:00 UTC.
- Danish `TX14`: local midnight → next local midnight, converted with `Europe/Copenhagen`; DST transition days span 23 or 25 hours.

Matched source-day records add Danish local-calendar days, ASTA days in fixed CET and flagged GSOD reporting days. [Source-day rules](../../../data2/cleaning_audit/verified_observation_window_override_rules.csv) document the evidence; apply only the [verified days](../../../data2/cleaning_audit/verified_observation_window_overrides.csv).

GSOD hourly fallback maxima cover the UTC reporting day. `window_sampling = "hourly_or_synoptic_samples"` marks the maximum of available samples; their exact times and completeness still need checking when comparing with ERA5. Other mapped rows use `provider_daily_maximum`.

`window_start_utc`, `window_end_utc` and `window_hours` describe the interval. `window_rule_id` identifies the evidence. Leave unsupported boundaries null with an explicit `window_status`; retain every study-period observation. ECA&D's existing date shifts are already included.

Finnish and Estonian archive checks cover winter and summer in 2020 and 2025. A dated Romanian temperature map and Serbia’s 2021 yearbook establish their date labels. Supporting comparisons: [Finland](../../../data2/cleaning_audit/fmi_window_verification.csv), [Estonia](../../../data2/cleaning_audit/estonia_window_verification.csv), [Romania](../../../data2/cleaning_audit/romania_window_verification.csv).

The tables and unresolved-source audit below cover only this study period. These are observation windows; availability of the required ERA5 hours is checked when matching.

Definitions: [ECA&D elements](https://www.ecad.eu/dailydata/datadictionaryelement.php), [date-shift documentation](https://knmi-ecad-assets-prd.s3.amazonaws.com/documents/atbd.pdf). Implementation: [observation_windows.py](../observation_windows.py).

In [10]:
from py.preprocessing.observation_windows import (
    attach_utc_windows, observation_window_rules, verified_window_rules,
)

window_rules = observation_window_rules()
provider_rules = verified_window_rules()
provider_rules.filter(
    (pl.col("valid_from") <= era5_last_date) & (pl.col("valid_to") >= era5_first_date)
).select("country_code", "observation_window", "valid_from", "valid_to", "time_zone")

country_code,observation_window,valid_from,valid_to,time_zone
str,str,i32,i32,str
"""SE""","""TX2""",0,99991231,"""UTC"""
"""SE""","""TX2""",0,99991231,"""UTC"""
"""NO""","""TX2""",0,99991231,"""UTC"""
"""FI""","""TX2""",0,99991231,"""UTC"""
"""AT""","""TX20""",20030101,99991231,"""UTC"""
…,…,…,…,…
"""RU""","""TX2""",0,99991231,"""UTC"""
"""FI""","""TX1""",20200101,20250831,"""UTC"""
"""EE""","""TX2""",20200101,20250831,"""UTC"""


In [11]:
station_data_with_windows = attach_utc_windows(study_station_data, source_geography)

window_counts = (
    station_data_with_windows.group_by("observation_window", "window_status", "window_sampling")
    .agg(
        pl.len().alias("daily_records"),
        pl.col("STAID").n_unique().alias("station_series"),
        pl.col("SOUID").n_unique().alias("sources"),
    )
    .sort("daily_records", descending=True)
)
assert station_data_with_windows.height == study_station_data.height
assert window_counts["daily_records"].sum() == study_station_data.height

source_window_counts = (
    station_data_with_windows.group_by("STAID", "SOUID", "observation_window", "window_status", "window_rule_id")
    .agg(
        pl.len().alias("daily_records"),
        pl.col("DATE").min().alias("first_date"),
        pl.col("DATE").max().alias("last_date"),
    )
    .join(source_geography.select(
        "STAID", "SOUID", "source_metadata_id", "source_name", "country_code",
        "provider_id", "provider_name", "source_lat", "source_lon",
    ), on=["STAID", "SOUID"], how="left", validate="m:1")
)
unresolved_windows = (
    source_window_counts.filter(pl.col("window_status") != "mapped")
    .group_by("country_code", "provider_id", "provider_name", "observation_window", "window_status")
    .agg(
        pl.col("daily_records").sum(), pl.col("SOUID").n_unique().alias("sources"),
        pl.col("first_date").min(), pl.col("last_date").max(),
    ).sort("daily_records", descending=True)
)

audit_dir = data_dir / "cleaning_audit"
audit_dir.mkdir(parents=True, exist_ok=True)
window_rules.write_csv(audit_dir / "preprocessing_window_rules.csv")
window_counts.write_csv(audit_dir / "preprocessing_window_counts.csv")
source_window_counts.write_csv(audit_dir / "preprocessing_window_sources.csv")
unresolved_windows.write_csv(audit_dir / "preprocessing_unresolved_windows.csv")

with pl.Config(tbl_rows=40):
    display(window_counts)

observation_window,window_status,window_sampling,daily_records,station_series,sources
cat,cat,cat,u32,u32,u32
"""TX3""","""mapped""","""provider_daily_maximum""",4386940,2399,2090
"""TX2""","""mapped""","""provider_daily_maximum""",3389320,1893,1366
"""TX20""","""mapped""","""provider_daily_maximum""",764745,586,381
"""TX7""","""mapped""","""provider_daily_maximum""",416200,784,489
"""TX9""","""mapped""","""provider_daily_maximum""",252346,161,148
"""TX14""","""mapped""","""provider_daily_maximum""",120547,59,53
"""TX21""","""mapped""","""provider_daily_maximum""",91899,236,234
"""TX6""","""mapped""","""provider_daily_maximum""",86966,43,42
"""TX1""","""interval_unknown""","""unknown""",65319,55,54


In [12]:
window_coverage = pl.DataFrame({
    "first_date": [era5_first_date],
    "last_date": [era5_last_date],
    "daily_records": [station_data_with_windows.height],
    "mapped_records": [station_data_with_windows.select((pl.col("window_status") == "mapped").sum()).item()],
    "sampled_reporting_day_records": [station_data_with_windows.select((pl.col("window_sampling") == "hourly_or_synoptic_samples").sum()).item()],
}).with_columns(
    (pl.col("daily_records") - pl.col("mapped_records")).alias("unresolved_records"),
    (100 * pl.col("mapped_records") / pl.col("daily_records")).alias("mapped_percent"),
)
window_coverage.write_csv(audit_dir / "preprocessing_window_coverage.csv")
display(window_coverage)
with pl.Config(tbl_rows=40):
    display(unresolved_windows)

first_date,last_date,daily_records,mapped_records,sampled_reporting_day_records,unresolved_records,mapped_percent
i64,i64,i64,i64,i64,i64,f64
20200101,20250831,9840278,9651857,34965,188421,98.085207


country_code,provider_id,provider_name,observation_window,window_status,daily_records,sources,first_date,last_date
str,i32,str,cat,cat,u32,u32,i32,i32
"""LU""",82,"""Andrew Ferrone""","""TX13""","""interval_unknown""",57030,34,20200101,20241116
"""LV""",23,"""Lubova Pirozenoka""","""TX19""","""sampling_unknown""",48134,21,20200101,20250731
"""GR""",83,"""GSOD""","""TX1""","""interval_unknown""",19613,19,20200101,20250227
"""AL""",83,"""GSOD""","""TX1""","""interval_unknown""",14213,9,20200101,20250227
"""BG""",83,"""GSOD""","""TX1""","""interval_unknown""",13653,8,20200101,20250228
"""ME""",68,"""Angel Marcev""","""TX17""","""date_anchor_unknown""",11410,6,20200101,20250831
"""HR""",6,"""Tanja Likso""","""TX1""","""interval_unknown""",9388,13,20200101,20211231
"""LV""",23,"""Lubova Pirozenoka""","""TX1""","""interval_unknown""",8086,4,20200101,20250731
"""HU""",13,"""Monika Lakatos""","""TX12""","""date_anchor_unknown""",4352,3,20200101,20250831


In [13]:
station_data_with_windows.filter(pl.col("window_status") == "mapped").unique(
    subset=["window_rule_id"], keep="first"
).select(
    "STAID", "SOUID", "DATE", "window_rule_id", "window_sampling",
    "window_start_utc", "window_end_utc", "window_hours",
).sort("window_rule_id")

STAID,SOUID,DATE,window_rule_id,window_sampling,window_start_utc,window_end_utc,window_hours
i32,i32,i32,cat,cat,"datetime[μs, UTC]","datetime[μs, UTC]",f64
217,683,20200101,"""anm_tx2_modern""","""provider_daily_maximum""",2019-12-31 18:00:00 UTC,2020-01-01 18:00:00 UTC,24.0
228,5697,20200101,"""arso_tx8""","""provider_daily_maximum""",2019-12-31 20:00:00 UTC,2020-01-01 20:00:00 UTC,24.0
280,121002,20241117,"""asta_cet_day""","""provider_daily_maximum""",2024-11-16 23:00:00 UTC,2024-11-17 23:00:00 UTC,24.0
107,71532,20200101,"""dmi_archive_day""","""provider_daily_maximum""",2019-12-31 23:00:00 UTC,2020-01-01 23:00:00 UTC,24.0
106,142264,20200101,"""dmi_tx14""","""provider_daily_maximum""",2019-12-31 23:00:00 UTC,2020-01-01 23:00:00 UTC,24.0
…,…,…,…,…,…,…,…
32,100,20200101,"""meteofrance_tx6""","""provider_daily_maximum""",2020-01-01 06:00:00 UTC,2020-01-02 06:00:00 UTC,24.0
262,813,20210101,"""rhms_serbia_tx8_2021""","""provider_daily_maximum""",2020-12-31 20:00:00 UTC,2021-01-01 20:00:00 UTC,24.0
2,36438,20200101,"""smhi_tx2""","""provider_daily_maximum""",2019-12-31 18:00:00 UTC,2020-01-01 18:00:00 UTC,24.0


## 7. Save the selected observations

Save the 2020–August 2025 observations through UTC-window mapping, with source coordinates, provider metadata, sampling labels and temperatures in °C. Keep unresolved windows as null timestamps with their status.

Output: `data2/selected_station_observations.parquet` (Zstandard compression). The audit manifest records row counts, input hashes and the notebook's source hash.

In [14]:
from py.preprocessing.save_selected_observations import save_selected_observations

selected_observations_file, selected_observations_manifest = save_selected_observations(
    station_data_with_windows, source_geography
)
pl.DataFrame({
    "file": [str(selected_observations_file)],
    "daily_records": [selected_observations_manifest["daily_records"]],
    "mapped_records": [selected_observations_manifest["mapped_records"]],
    "unresolved_records": [selected_observations_manifest["unresolved_records"]],
    "size_mib": [selected_observations_manifest["bytes"] / 1024**2],
})

file,daily_records,mapped_records,unresolved_records,size_mib
str,i64,i64,i64,f64
"""/home2/lobib/gen_hack/data2/se…",9840278,9651857,188421,36.642305
